# 01 - Download datasets once, store in Drive
Runtime: **CPU**. Safe to re-run: skips what is already in Drive.
**Already ran it? You do not need to re-run. Just replace the Kaggle cell and clear outputs (see below).**

| Dataset | How | Notes |
|---|---|---|
| DeepPCB | automatic (GitHub) | 1,500 test/template pairs, 6 defect classes, **11 boards (groups)**. Research-use-only licence: check before any commercial/patent use |
| Gerber samples (IEEE DataPort) | **manual** upload | needs a free IEEE login |
| Kaggle / HF | optional cells | token in Colab Secrets |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
D = '/content/drive/MyDrive/pcbrisk'
sys.path.insert(0, f'{D}/code')
from pcbrisk_utils import *
seed_all(42)

Mounted at /content/drive


In [ ]:
import os
raw = f'{L}/raw'; os.makedirs(raw, exist_ok=True)
zip_drive = f'{D}/00_raw/deeppcb_main.zip'

if not os.path.exists(zip_drive):
    sh(f'git clone --depth 1 https://github.com/tangsanli5201/DeepPCB {raw}/DeepPCB')
    sh(f'cd {raw} && zip -qr deeppcb_main.zip DeepPCB -x "DeepPCB/.git/*"')
    safe_copy(f'{raw}/deeppcb_main.zip', zip_drive)
    print(append_checksum(zip_drive))
else:
    print('already in Drive:', zip_drive)

src = f'{D}/00_raw/SOURCES.md'
if not os.path.exists(src) or 'DeepPCB' not in open(src).read():
    from datetime import date
    with open(src, 'a') as f:
        f.write(f"- DeepPCB | https://github.com/tangsanli5201/DeepPCB | fetched {date.today()} | README: research use only; defects partly artificial; cite Tang et al. 2019\n")

already in Drive: /content/drive/MyDrive/pcbrisk/00_raw/deeppcb_main.zip


In [ ]:
if not os.path.exists(f'{L}/raw/DeepPCB'):
    sh(f'unzip -q {zip_drive} -d {L}/raw')
import glob
tests = glob.glob(f'{L}/raw/DeepPCB/PCBData/group*/*/*_test.jpg')
temps = glob.glob(f'{L}/raw/DeepPCB/PCBData/group*/*/*_temp.jpg')
print(len(tests), 'test imgs,', len(temps), 'template imgs')   # 1500 / 1501 is normal (one stray template)
print(len(glob.glob(f'{L}/raw/DeepPCB/PCBData/group*')), 'boards (groups)')   # 11

1500 test imgs, 1501 template imgs
11 boards (groups)


## Gerber samples / other manual downloads
1. Download the zip in your browser (logged in).
2. Drag it into `MyDrive/pcbrisk/00_raw/` on drive.google.com (not the Colab file panel).
3. Run the next cell to checksum it, then fill in `SOURCES.md` (URL, date, licence, citation).

In [ ]:
import glob
for z in glob.glob(f'{D}/00_raw/*.zip') + glob.glob(f'{D}/00_raw/*.tar*'):
    print(append_checksum(z))
print(open(f'{D}/00_raw/checksums.txt').read())

5654be20ec5b84c10e7b234ce2fce78b0c046251dbca513b9b6c4255256af3ca  deeppcb_main.zip
5654be20ec5b84c10e7b234ce2fce78b0c046251dbca513b9b6c4255256af3ca  deeppcb_main.zip



## Optional: Kaggle / Hugging Face
**Secret NAMES go in `userdata.get(...)`, never the secret VALUES.** Colab left bar -> key icon -> add secrets named `KAGGLE_USERNAME` and `KAGGLE_KEY`, toggle notebook access ON, then run:

In [ ]:
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
sh('pip -q install kaggle')
slug = 'owner/dataset-name'          # <- change
sh(f'kaggle datasets download -d {slug} -p {L}/raw --unzip')
# then: tar -> safe_copy to 00_raw -> append_checksum

CalledProcessError: Command 'kaggle datasets download -d owner/dataset-name -p /content/raw --unzip' returned non-zero exit status 1.

In [ ]:
# Hugging Face (secret named HF_TOKEN)
# from huggingface_hub import snapshot_download
# p = snapshot_download('org/dataset', repo_type='dataset', local_dir=f'{L}/raw/hf_ds', token=userdata.get('HF_TOKEN'))